# 🔬 Notebook Analisis Kritis: AI Adoption & Productivity Analysis (2021–2026)
**Project:** Data Analyst Portfolio - Deep Dive & Critical Insights  
**Dataset Files:** `ai_adoption_productivity_2021_2026.csv` & `user_level_ai_adoption.csv`  
**Data Analyst Author:** Rafli A.  
**Dataset Source:** [Global AI Usage and Productivity - Kaggle](https://www.kaggle.com/datasets/ashyou09/global-ai-usage-and-productivity) 

---

## 📌 Latar Belakang, Sitasi & Tujuan Analisis Kritis
Notebook ini dirancang untuk menyajikan **analisis kritis dan mendalam** terhadap data adopsi AI dan produktivitas pengguna.

> **Dataset Credit & Attribution:**  
> Dataset yang digunakan dalam proyek analisis data ini berasal dari sumber open source:  
> 🔗 [Global AI Usage and Productivity Dataset - Kaggle](https://www.kaggle.com/datasets/ashyou09/global-ai-usage-and-productivity)

Analisis ini tidak hanya menyajikan angka agregat, tetapi juga menguji validitas hipotesis, mendeteksi anomali/paradoks data, mengevaluasi potensi bias *self-reporting*, serta mengukur variabel penentu produktivitas berbasis pemodelan statistik dan *unsupervised clustering*.

### 1. Inisialisasi Environment & Input Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
import os

# Configuration
plt.style.use('ggplot')
pd.set_option('display.max_columns', None)

# Load Datasets
df_macro = pd.read_csv('../data/ai_adoption_productivity_2021_2026.csv')
df_micro = pd.read_csv('../data/user_level_ai_adoption.csv')

print(f"Macro Dataset Shape: {df_macro.shape}")
print(f"Micro Dataset Shape: {df_micro.shape}")

Macro Dataset Shape: (402, 6)
Micro Dataset Shape: (15000, 10)


### 2. Audit Kualitas Data & Pemeriksaan Distribusi (Data Quality & Anomaly Detection)

Pada bagian ini kita mengevaluasi kecenderungan sentral, pencilan (*outliers*), dan skewness dari variabel numerik utama untuk mendeteksi apakah data berdistribusi normal atau memiliki skewness ekstrem.

In [2]:
# Numerical Feature Statistics & Skewness Analysis
num_cols = ['Experience_Years', 'Daily_Token_Usage', 'Tasks_Automated_Per_Week', 'Productivity_Gain_Percent']
summary_df = df_micro[num_cols].describe().T
summary_df['skewness'] = df_micro[num_cols].skew()
summary_df['kurtosis'] = df_micro[num_cols].kurt()

print("=== STATISTICAL SUMMARY & DISTRIBUTIONS ===")
print(summary_df[['count', 'mean', 'std', 'min', '50%', 'max', 'skewness', 'kurtosis']])

=== STATISTICAL SUMMARY & DISTRIBUTIONS ===
                             count         mean          std    min     50%  \
Experience_Years           15000.0    13.013400     7.215642    1.0    13.0   
Daily_Token_Usage          15000.0  8958.312200  8449.890624  411.0  7336.0   
Tasks_Automated_Per_Week   15000.0     1.913267     1.176196    1.0     2.0   
Productivity_Gain_Percent  15000.0    11.215827    11.578100    0.3     8.1   

                               max  skewness   kurtosis  
Experience_Years              25.0  0.000240  -1.217810  
Daily_Token_Usage          58989.0  2.515177   8.148892  
Tasks_Automated_Per_Week      12.0  2.548621  11.214355  
Productivity_Gain_Percent     84.9  2.767237  10.445975  


#### 💡 Catatan Kritis Auditor Data:
1. **Right-Skewness pada Token Usage:** `Daily_Token_Usage` memiliki *skewness* positif yang signifikan, di mana nilai median (7.336 token) jauh berada di bawah rata-rata (8.958 token) dan maksimum mencapai 58.989 token. Ini mengindikasikan keberadaan kelompok *Power Users* yang mengonsumsi token secara sangat intensif.
2. **Productivity Gain Disparities:** Variabel `Productivity_Gain_Percent` memiliki rentang dari 5% hingga 84.9%. Variansi yang lebar ini memerlukan pembongkaran lebih lanjut berdasarkan kategori tools AI.

### 3. Pengujian Hipotesis Kritis & Pembongkaran Paradoks Bisnis

#### Hipotesis A: Token Usage vs Productivity Gain (Korelasi vs Diminishing Returns)
Apakah peningkatan konsumsi token secara otomatis menghasilkan kenaikan produktivitas linier?

In [3]:
# Pearson Correlation Calculation
r_token = df_micro['Daily_Token_Usage'].corr(df_micro['Productivity_Gain_Percent'])
r_tasks = df_micro['Tasks_Automated_Per_Week'].corr(df_micro['Productivity_Gain_Percent'])
r_exp = df_micro['Experience_Years'].corr(df_micro['Productivity_Gain_Percent'])

print(f"Correlation Token Usage vs Productivity Gain : r = {r_token:.4f}")
print(f"Correlation Tasks Automated vs Productivity Gain: r = {r_tasks:.4f}")
print(f"Correlation Experience Years vs Productivity Gain: r = {r_exp:.4f}")

Correlation Token Usage vs Productivity Gain : r = 0.8816
Correlation Tasks Automated vs Productivity Gain: r = 0.5548
Correlation Experience Years vs Productivity Gain: r = -0.0117


#### Hipotesis B: Jurang Efisiensi antara Specialty Tools vs Generalist Tools
Mengapa *DeepSeek* dan *GitHub Copilot* memberikan *Productivity Gain* rata-rata di atas **40%**, sementara tools seperti *ChatGPT*, *Claude*, dan *Gemini* berada di kisaran **10%**?

In [4]:
# Group analysis by Primary AI Tool
tool_perf = df_micro.groupby('Primary_AI_Tool').agg(
    Users=('User_ID', 'count'),
    Avg_Tokens=('Daily_Token_Usage', 'mean'),
    Avg_Tasks=('Tasks_Automated_Per_Week', 'mean'),
    Avg_Gain=('Productivity_Gain_Percent', 'mean'),
    Median_Gain=('Productivity_Gain_Percent', 'median')
).sort_values('Avg_Gain', ascending=False)

print("=== PERFORMANCE MATRIX BY PRIMARY AI TOOL ===")
print(tool_perf)

=== PERFORMANCE MATRIX BY PRIMARY AI TOOL ===
                    Users    Avg_Tokens  Avg_Tasks   Avg_Gain  Median_Gain
Primary_AI_Tool                                                           
DeepSeek              189  32867.010582   4.465608  42.565608         38.0
GitHub Copilot        918  32364.824619   4.127451  40.110566         36.3
Perplexity           1949   8007.466906   1.685993  10.139046          8.6
Claude (Anthropic)   2980   8040.822819   1.691275  10.119899          8.8
ChatGPT (OpenAI)     5430   8003.693923   1.679742   9.999761          8.9
Gemini (Google)      1483   7977.279838   1.694538   9.888672          8.8
Midjourney           2051   1751.994149   2.001950   2.188737          1.9


#### 🔍 Analisis Kritis Kunci:
- **Specialist Coding Tools vs Generalist Text Tools:** *GitHub Copilot* dan *DeepSeek* berfokus pada eksekusi sintaksis koding kompleks dan otomatisasi tugas teknis bernilai tinggi, sehingga menghasilkan penghematan jam kerja yang jauh lebih terukur dibanding generasi teks umum (*ChatGPT*, *Claude*, *Gemini*).
- **Kelemahan Tools Visual (Midjourney):** *Midjourney* mencatatkan *productivity gain* terendah (rata-rata 2.19%). Hal ini wajar karena pembuatan aset kreatif visual membutuhkan iterasi eksploratif dan revisi artistik manusia yang tidak serta merta memangkas durasi kerja secara otomatis.

#### Hipotesis C: Regresi Multivariat OLS, Tabel Summary Inferensial (HC3 Robust SE) & Diagnostik Residual

Untuk memastikan bahwa klaim signifikansi statistik didukung secara inferensial, pemodelan OLS dieksekusi menggunakan `statsmodels.formula.api.ols()` dengan **Heteroskedasticity Robust Standard Errors (HC3)**.

Model mengontrol `Daily_Token_Usage`, `Tasks_Automated_Per_Week`, `Experience_Years`, `Primary_AI_Tool` (Ref: `ChatGPT (OpenAI)`), dan `Industry` (Ref: `Creative & Design`).

In [5]:
# Pemodelan OLS Regresi Inferensial dengan Statsmodels (HC3 Robust Standard Errors)
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import jarque_bera

df_micro['Industry_Clean'] = df_micro['Industry'].str.replace(' & ', '_').str.replace(' ', '_')
formula_a = "Productivity_Gain_Percent ~ Daily_Token_Usage + Tasks_Automated_Per_Week + Experience_Years + C(Primary_AI_Tool, Treatment(reference='ChatGPT (OpenAI)')) + C(Industry_Clean, Treatment(reference='Creative_Design'))"

model_hc3 = smf.ols(formula=formula_a, data=df_micro).fit(cov_type='HC3')

# Diagnostic Tests
residuals = model_hc3.resid
bp_test = het_breuschpagan(residuals, model_hc3.model.exog)
jb_test = jarque_bera(residuals)

print("=== STATSMODELS OLS INFERENTIAL SUMMARY TABLE (HC3 Robust Standard Errors) ===")
print(model_hc3.summary())

print("\n=== RESIDUAL DIAGNOSTICS & HETEROSKEDASTICITY TESTS ===")
print(f"Breusch-Pagan LM Stat   : {bp_test[0]:.4f}, p-value: {bp_test[1]:.4e} (Significant Heteroskedasticity -> HC3 SE Required)")
print(f"Jarque-Bera Normality   : {jb_test[0]:.4f}, p-value: {jb_test[1]:.4e} (Right-skewed residual distribution)")
print(f"Residual Std Error (RSE): {np.sqrt(model_hc3.mse_resid):.4f}")
print(f"Adjusted R²             : {model_hc3.rsquared_adj:.4f}")
print(f"AIC / BIC               : {model_hc3.aic:.2f} / {model_hc3.bic:.2f}")

=== STATSMODELS OLS INFERENTIAL SUMMARY TABLE (HC3 Robust Standard Errors) ===
                                OLS Regression Results                               
Dep. Variable:     Productivity_Gain_Percent   R-squared:                       0.807
Model:                                   OLS   Adj. R-squared:                  0.807
Method:                        Least Squares   F-statistic:                     2068.
No. Observations:                      15000   AIC:                         9.136e+04
Df Residuals:                          14985   BIC:                         9.148e+04
Df Model:                                 14   Covariance Type:                   HC3
                                                                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------------------------------------------------
Intercept          

#### ⚠️ Catatan Kritis Validasi Inferensial & Diagnostik Residual:
1. **Uji Heteroskedastisitas Breusch-Pagan:** Nilai $	ext{LM Stat} = 5183.02 (p < 0.0001)$ mengonfirmasi keberadaan varians residual yang tidak konstan (*heteroskedasticity*). Penggunaan **HC3 Robust Standard Errors** menjamin bahwa nilai *standard error*, *z-statistics*, dan *p-values* bebas dari bias heteroskedastisitas.
2. **Signifikansi Koefisien Terkontrol (P-Values & 95% CI):**
   - **`Daily_Token_Usage`**: $eta = +0.001071$, Robust SE $= 0.000015$, $z = 72.172$, $p < 0.0001$, 95% CI $= [0.001042, 0.001100]$. Terbukti signifikan secara statistik.
   - **`Tasks_Automated_Per_Week`**: $eta = +1.9926$, Robust SE $= 0.0738$, $z = 27.014$, $p < 0.0001$, 95% CI $= [1.848, 2.137]$. Terbukti signifikan secara statistik.
   - **`Experience_Years`**: $eta = -0.0115$, Robust SE $= 0.0056$, $z = -2.033$, $p = 0.042$, 95% CI $= [-0.0225, -0.0004]$. Efek marjinal sangat kecil.
3. **Normalitas Residual (Jarque-Bera Test):** Nilai $	ext{JB Stat} = 11214.40 (p < 0.0001)$ dengan *Kurtosis* = $7.21$ menunjukkan ekor distribusi residual cenderung *heavy-tailed/right-skewed*, yang wajar terjadi pada data penggunaan token dan efisiensi.

### 4. Analisis Paritas Senioritas Pekerja (Independensi Gain terhadap Senioritas vs Baseline Pre-Adopsi)

Apakah AI memberikan peningkatan produktivitas yang berbeda secara signifikan antara pekerja Junior, Mid-Level, Senior, dan Veteran?
Untuk menguji klaim ini secara objektif, kita menguji **95% Confidence Intervals**, **Uji Hipotesis ANOVA (Analysis of Variance)**, serta **Model Interaksi ($	ext{Experience} 	imes 	ext{Primary\_AI\_Tool}$)**.

In [6]:
# Dynamic One-Way ANOVA Testing using scipy.stats.f_oneway & Eta-Squared Effect Size
from scipy.stats import f_oneway

bins_exp = [-1, 3, 8, 15, 100]
labels_exp = ['Junior (0-3 yrs)', 'Mid-Level (4-8 yrs)', 'Senior (9-15 yrs)', 'Veteran (>15 yrs)']
df_micro['Experience_Group'] = pd.cut(df_micro['Experience_Years'], bins=bins_exp, labels=labels_exp)

exp_summary = df_micro.groupby('Experience_Group', observed=False).agg(
    User_Count=('User_ID', 'count'),
    Avg_Daily_Tokens=('Daily_Token_Usage', 'mean'),
    Avg_Tasks_Automated=('Tasks_Automated_Per_Week', 'mean'),
    Avg_Productivity_Gain=('Productivity_Gain_Percent', 'mean'),
    Std_Dev=('Productivity_Gain_Percent', 'std')
)
exp_summary['SE'] = exp_summary['Std_Dev'] / np.sqrt(exp_summary['User_Count'])
exp_summary['CI95_Low'] = exp_summary['Avg_Productivity_Gain'] - 1.96 * exp_summary['SE']
exp_summary['CI95_High'] = exp_summary['Avg_Productivity_Gain'] + 1.96 * exp_summary['SE']

# Dynamic One-Way ANOVA & Eta-Squared Effect Size
group_data = [df_micro[df_micro['Experience_Group'] == eg]['Productivity_Gain_Percent'].values for eg in labels_exp]
overall_mean = df_micro['Productivity_Gain_Percent'].mean()
ss_between = sum(len(g) * (np.mean(g) - overall_mean)**2 for g in group_data)
ss_within = sum(sum((x - np.mean(g))**2 for x in g) for g in group_data)
eta_squared = ss_between / (ss_between + ss_within)

f_stat, p_val_anova = f_oneway(*group_data)

print("=== DYNAMIC ANOVA TESTING & ETA-SQUARED EFFECT SIZE ===")
print(exp_summary[['User_Count', 'Avg_Daily_Tokens', 'Avg_Productivity_Gain', 'CI95_Low', 'CI95_High']])
print(f"\nDynamic ANOVA F-statistic : F = {f_stat:.4f} (df1=3, df2={len(df_micro)-4})")
print(f"Dynamic ANOVA p-value     : p = {p_val_anova:.6f} (Fail to reject H0 of equal group means)")
print(f"Effect Size Eta-Squared   : η² = {eta_squared:.6f} (Negligible effect size < 0.01)")

=== DYNAMIC ANOVA TESTING & ETA-SQUARED EFFECT SIZE ===
                     User_Count  Avg_Daily_Tokens  Avg_Productivity_Gain  CI95_Low  CI95_High
Experience_Group                                                                              
Junior (0-3 yrs)           1763       8885.018151              11.248780  10.722684  11.774876
Mid-Level (4-8 yrs)        3055       9069.153846              11.417741  11.003664  11.831818
Senior (9-15 yrs)          4129       9005.970695              11.274691  10.918451  11.630931
Veteran (>15 yrs)          6053       8891.207335              11.064167  10.773010  11.355324

Dynamic ANOVA F-statistic : F = 0.6962 (df1=3, df2=14996)
Dynamic ANOVA p-value     : p = 0.554241 (Fail to reject H0 of equal group means)
Effect Size Eta-Squared   : η² = 0.000139 (Negligible effect size < 0.01)


#### 💡 Temuan Kritis & Evaluasi Metodologi Paritas Senioritas:
1. **Uji Dinamis ANOVA & Effect Size Eta-Squared ($\eta^2$):** Hasil pengujian `scipy.stats.f_oneway()` secara dinamis menghasilkan $F = 0.6962$ ($p = 0.554241$) dan **Eta-Squared ($\eta^2 = 0.000139$)**. Nilai $\eta^2 < 0.01$ mengonfirmasi bahwa **besarnya perbedaan produktivitas antar tingkat senioritas adalah sangat tidak signifikan (abaikan/negligible)**. Sel Selang Kepercayaan (95% CI) seluruh kelompok saling tumpang-tindih (*overlapping*) pada rentang 10.7%–11.8%.
2. **Keterbatasan Klaim "Productivity Equalizer":** Nilai $p = 0.554$ dan $\eta^2 pprox 0.00014$ **belum membuktikan** bahwa teknologi AI "menyamakan" (*equalize*) produktivitas junior dan senior secara absolut.
   - Untuk membuktikan efek pemerataan (*equalizing effect*), diperlukan data observasi **sebelum adopsi (Pre-AI Baseline)** dan **setelah adopsi (Post-AI)**.
   - Data observational cross-sectional saat ini membuktikan bahwa **persentase manfaat efisiensi dari adopsi AI bersifat independen dari tingkat senioritas pekerja** (*Seniority-Independent AI Productivity Gains*).

### 5. Rekomendasi Strategis & Implikasi Bisnis (Strategic Recommendations)

Berdasarkan analisis kritis di atas, berikut adalah 4 rekomendasi strategis bagi pimpinan organisasi/bisnis:

1. **Alokasi Investasi AI Terarah (Targeted AI Tooling):** Prioritaskan pengadaan tools AI terpesialisasi (*Domain-Specific AI Tools*) seperti *Copilot* dan *DeepSeek* untuk divisi teknis koding/analitis daripada hanya menyediakannya sebagai lisensi umum.
2. **Tata Kelola Kuota Token Berbasis Perangkat & Peran (Role & Tool-Aware Token Governance):** Menghindari asumsi naif bahwa menaikkan kuota token secara umum akan otomatis meningkatkan produktivitas ($r = 0.88$ sebagian besar terjelaskan oleh kategori perangkat). Kuota token harus dialokasikan secara selektif berdasarkan kebutuhan alur kerja spesifik (misal: pengembang software dengan *Copilot/DeepSeek* membutuhkan batas kuota jauh lebih tinggi dibanding pengguna teks umum).
3. **Standardisasi Otomatisasi Alur Kerja:** Dorong integrasi alur kerja di mana AI tidak sekadar digunakan untuk tanya-jawab (*Q&A*), tetapi digunakan untuk otomatisasi alur kerja berulang (*task automation*).
4. **Mitigasi Bias Self-Reporting:** Untuk riset internal lanjutan, disarankan menggabungkan variabel persepsi produktivitas dengan metrik kinerja objektif (*seperti Pull Request completion time, ticket closure rate, atau project turnaround time*).